### Load Libraries

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.stem import PorterStemmer

c:\Users\rirbouh\AppData\Local\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read Loughran Mcdonald's Lexicon 

(lexicon is updated to only get Negative Neutral and Positive sentiment words)

In [ ]:
df_lm= pd.read_csv("LM-SA-2020.csv", sep=';')
df_lm

### Generate LM's Class

In [ ]:
class LoughranMcDonaldDict:
    def __init__(self, csv_file_path):
        self.stemmer = PorterStemmer()
        try:
            df = pd.read_csv(csv_file_path, sep=";")
            # Convert to stems when loading
            self.positive_words = set(df[df['sentiment'] == 'Positive']['word'].str.lower().apply(self.stemmer.stem))
            self.negative_words = set(df[df['sentiment'] == 'Negative']['word'].str.lower().apply(self.stemmer.stem))
            self.neutral_words = set(df[df['sentiment'] == 'Neutral']['word'].str.lower().apply(self.stemmer.stem))
        except Exception as e:
            print(f"Error processing dictionary: {e}")
            self._load_basic_dict()

    def _load_basic_dict(self):
        """Dictionnaire de base en cas d'erreur"""
        self.positive_words = {'good', 'excellent', 'great', 'successful', 'positive'}
        self.negative_words = {'bad', 'error', 'problem', 'fail', 'negative', 'urgent', 'crisis'}
        self.neutral_words = {'uncertain', 'maybe', 'possible', 'likely'}

    def analyze_text(self, text):
        # Stem the words before matching
        words = re.findall(r'\b\w+\b', text.lower())
        stemmed_words = [self.stemmer.stem(word) for word in words]
        
        positive_count = sum(1 for word in stemmed_words if word in self.positive_words)
        negative_count = sum(1 for word in stemmed_words if word in self.negative_words)
        neutral_count = sum(1 for word in stemmed_words if word in self.neutral_words)
        
        total_words = len(words)
        if total_words == 0:
            return {
                'score': 0,
                'positive_count': 0,
                'negative_count': 0,
                'neutral_count': 0,
                'total_words': 0
            }
        
        # Calculate normalized scores
        positive_score = positive_count / total_words
        negative_score = negative_count / total_words
        neutral_score = neutral_count / total_words
        
        # Final dictionnary score (take neutral words as slightly negative)
        dict_score = positive_score - negative_score - (0.1 * neutral_score)
        
        return {
            'score': dict_score,
            'positive_count': positive_count,
            'negative_count': negative_count,
            'neutral_count': neutral_count,
            'total_words': total_words
        }


### Create your hybrid sentiment analysis 

In [ ]:
class HybridSentimentAnalyzer:
    def __init__(self, lm_dict_path, model_weight=1, dict_weight=0.5):
        # Load model
  
        self.tokenizer = AutoTokenizer.from_pretrained("roberta-base-sentiment-latest")
        self.model = AutoModelForSequenceClassification.from_pretrained("roberta-base-sentiment-latest")

        self.lm_dict = LoughranMcDonaldDict(lm_dict_path)
        
        # weights for combination of model and dictionary
        self.model_weight = model_weight
        self.dict_weight = dict_weight
        
        # model labels
        self.labels = ["NEGATIVE", "NEUTRAL", "POSITIVE"]

    
    def get_model_sentiment(self, text):
        """Get sentiment from RoBERTa"""
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, 
                               padding=True, max_length=512)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        
        # Convertir en score -1 à 1 (négatif à positif)
        negative_prob = predictions[0][0].item()
        neutral_prob = predictions[0][1].item()
        positive_prob = predictions[0][2].item()
        
        # Score du modèle (-1 à 1)
        model_score = positive_prob - negative_prob
        return model_score, predictions[0]
    
    def analyze(self, text):
        """Analyse texte"""
        # 1. Get model score
        model_score, model_probs = self.get_model_sentiment(text)
        
        # 2. Get dictionnary score
        dict_analysis = self.lm_dict.analyze_text(text)
        dict_score = dict_analysis['score']
        
        # 3. Combine weights
        final_score = (self.model_weight * model_score) + (self.dict_weight * dict_score)
        
        # 4. Get final sentiment
        if final_score > 0.15:
            sentiment = "POSITIVE"
        elif final_score < -0.15:
            sentiment = "NEGATIVE"
        else:
            sentiment = "NEUTRAL"
        
        return {
            'text': text,
            'final_sentiment': sentiment,
            'final_score': final_score,
            'model_score': model_score,
            'dict_score': dict_score,
            'model_probs': model_probs,
            'dict_details': dict_analysis,
            'breakdown': {
                'model_contribution': self.model_weight * model_score,
                'dict_contribution': self.dict_weight * dict_score
            }
        }



### Help function to get all words 

In [ ]:
def find_detected_words(text, lm_dict):
    """Fdetect words of each catégory in the text"""
    words = re.findall(r'\b\w+\b', text.lower())
    
    detected = {
        'Positive': [w for w in words if w in lm_dict.positive_words],
        'Negative': [w for w in words if w in lm_dict.negative_words],
        'Neutral': [w for w in words if w in lm_dict.neutral_words],
    }
    
    return detected

### Initialise model

In [ ]:
analyzer = HybridSentimentAnalyzer(r"Path/to/your/dictionnary/LM-SA-2020.csv", model_weight=1, dict_weight=0.5)

### Predict

In [ ]:
# Test 
text = '''Your text here'''

#Sample texte (replace text variable)
'''The recent earnings report from ABC Corporation has shown unexpected growth, exceeding analysts' expectations. Despite the challenges posed by the ongoing supply chain issues, the company managed to increase its revenue by 15% year-over-year.

Investors are optimistic about the future, as the management has provided strong guidance for the upcoming quarters. The stock price surged by 10% following the announcement, reflecting positive sentiment in the market.

However, some analysts caution that the inflationary pressures and rising interest rates could pose risks to the company’s margin in the long term. While the current performance is impressive, there are concerns about sustainability.

Overall, the sentiment surrounding ABC Corporation is mixed. Short-term outlooks appear bright, but uncertainties loom for the latter part of the fiscal year.

'''

print("\n=== ANALYSING ===")
result = analyzer.analyze(text)

print("=== RESULTS ===")
print(f"final Sentiment: {result['final_sentiment']}")
print(f"final Score: {result['final_score']:.4f}")
print(f"\nDetails:")
print(f"- Model Score: {result['model_score']:.4f}")
print(f"- Dictionnary Score: {result['dict_score']:.4f}")
print(f"\nContributions:")
print(f"- Model Contribution: {result['breakdown']['model_contribution']:.4f}")
print(f"- Dictionnary Contribution: {result['breakdown']['dict_contribution']:.4f}")

print(f"\n Detailed Dictionnary analysis")
print(f"- Posisve words found: {result['dict_details']['positive_count']}")
print(f"- Negative words found: {result['dict_details']['negative_count']}")
print(f"- Neutral words found: {result['dict_details']['neutral_count']}")
print(f"- Total Words Found: {result['dict_details']['total_words']}")

print(f"\nModel Probability:")
for i, label in enumerate(analyzer.labels):
    print(f"- {label}: {result['model_probs'][i]:.4f}")

# Afficher les mots détectés
detected_words = find_detected_words(text, analyzer.lm_dict)
print(f"\n=== DETECTED WORDS ===")
for category, words in detected_words.items():
    if words:
        print(f"{category.upper()}: {', '.join(set(words))}")
